# Lesson 6.3 — The Four Fundamental Subspaces
**Module 6 · Unit 6 · Lesson 23**

Split SVD directions by zero/nonzero gain: row space (moves tool) + null space (self-motion) in joint space; range (reachable) + left-null (impossible) in task space.

In [1]:
import numpy as np
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i]); M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def Jv_planar(P,T,q): return geometric_jacobian(P,T,q)[:2,:]
P2=[(0,0,1,0),(0,0,1,0)]; T2=["R","R"]
P3=[(0,0,1,0),(0,0,1,0),(0,0,0.6,0)]; T3=["R","R","R"]


## Redundant 3R: the null space is self-motion (J q̇_null = 0)

In [2]:
checks=[]
Jr=Jv_planar(P3,T3,np.array([0.3,0.5,-0.4])); U,S,Vt=np.linalg.svd(Jr); r=np.linalg.matrix_rank(Jr)
null=Vt[r:]            # right-singular vectors with zero gain
print("rank",r," null-space basis (joint self-motion):",np.round(null,3))
print("J @ null^T =",np.round(Jr@null.T,8).ravel())
checks.append(np.allclose(Jr@null.T,0,atol=1e-9))

rank 2  null-space basis (joint self-motion): [[-0.381  0.283  0.88 ]]
J @ null^T = [-0. -0.]


## Singular 2R: the left-null space is the impossible direction (Jᵀ u = 0)

In [3]:
Js=Jv_planar(P2,T2,np.array([0.4,0.0])); Us,Ss,_=np.linalg.svd(Js)
impossible=Us[:,1]     # left-singular vector with zero gain
print("impossible (left-null) direction:",np.round(impossible,3)," J^T @ it =",np.round(Js.T@impossible,8))
checks.append(np.allclose(Js.T@impossible,0,atol=1e-9))

impossible (left-null) direction: [0.921 0.389]  J^T @ it = [0. 0.]


## Map of the four subspaces

In [4]:
print("joint space: row space (moves tool) + null space (self-motion)")
print("task  space: range (reachable)      + left-null (impossible)")
print("J maps row space -> range with gains sigma_i; null -> 0; left-null unreachable")
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

joint space: row space (moves tool) + null space (self-motion)
task  space: range (reachable)      + left-null (impossible)
J maps row space -> range with gains sigma_i; null -> 0; left-null unreachable
All checks passed.
